# 🟡 KaizenStat — Intermediate Demo (15 min)

**Level:** Intermediate | **Time:** ~15 minutes | **Dataset:** Titanic

This notebook covers the **full 8-step pipeline** plus:
- Root-cause model debugging
- Production Trust Score
- Pipeline Confidence Score
- Understanding improvement suggestions

---
> **Prerequisites:** You've seen the Basic demo or know basic pandas/sklearn
>
> **What you'll learn:**
> - How to diagnose *why* your model fails (data vs model problem)
> - How to get a production readiness score
> - How to interpret improvement suggestions with priority levels
> - How to read feature importances and failure slices

In [ ]:
!pip install kaizenstat -q
print("✅ KaizenStat installed")

## Setup — Load and Inspect

We'll use the Titanic dataset but this time look at it more carefully before running the pipeline.

In [ ]:
import pandas as pd
import numpy as np
from kaizenstat import DataDoctor

# Load Titanic dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("=" * 50)
print("TITANIC DATASET OVERVIEW")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"\nTarget distribution (Survived):")
print(df['Survived'].value_counts())
print(f"\nImbalance ratio: {df['Survived'].value_counts(normalize=True).round(3).to_dict()}")
print(f"\nMissing values:")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"\nData types:")
print(df.dtypes)

## Full 8-Step Pipeline

We'll now run all 8 steps with detailed explanations of what each one tells us.

### Step 1 — fit()

In [ ]:
doctor = DataDoctor()
doctor.fit(df, target="Survived")

### Step 2 — health()

The health score tells us what percentage of the data is "production-ready".
Titanic has known issues: `Cabin` has 77% missing values, `Age` has 19.9% missing.

In [ ]:
health = doctor.health()
print(f"\nHealth Score: {health.score} / 100")
print("\n--- Understanding the score ---")
print("Cabin: 77% missing → major penalty")
print("Age: 19.9% missing → moderate penalty")
print("Embarked: 0.2% missing → minor penalty")

### Step 3 — validate()

Validation checks things health scoring misses:
- **Leakage:** Could `PassengerId` or `Name` be correlating with survival by accident?
- **Drift:** Are training and test distributions similar?
- **Multicollinearity:** Are `Pclass` and `Fare` too correlated?

In [ ]:
validation = doctor.validate()
print(f"\nIssues found: {len(validation.issues)}")

### Step 4 — fix(safe=True)

Preview mode shows you exactly what will happen before making any changes.

In [ ]:
# First — preview only, no changes applied
print("PREVIEW MODE — no changes yet")
print("-" * 40)
doctor.fix(safe=True, preview_only=True)
print(f"\nMissing values still: {df.isnull().sum().sum()}")

In [ ]:
# Now apply the fixes
print("APPLYING FIXES")
print("-" * 40)
fixed_df = doctor.fix(safe=True)
print(f"\nAfter fix: {fixed_df.isnull().sum().sum()} missing values")
print(f"Columns kept: {list(fixed_df.columns)}")

### Step 5 — train()

The benchmark shows **why** certain models win. On Titanic:
- Tree-based models (RF, GBM) typically outperform Logistic Regression
- Titanic is a classic case where feature engineering matters more than model choice

Watch the benchmark ranking carefully — it tells you which algorithm class fits your data.

In [ ]:
train_result = doctor.train(cv=5)

print(f"\n{'='*40}")
print("TRAINING RESULTS")
print(f"{'='*40}")
print(f"Best model:  {train_result.model_name}")
print(f"Task type:   {train_result.task}")
print(f"Test score:  {train_result.test_score:.4f}")
print(f"Train score: {train_result.train_score:.4f}")
print(f"Gap:         {train_result.train_score - train_result.test_score:.4f}")

### Step 6 — debug_model()

This is where KaizenStat is **uniquely valuable**.

Most tools show you accuracy. `debug_model()` shows you:
1. **Data vs model blame** — Is the low score because of bad data, or the wrong model choice?
2. **Feature importances** — Which features actually drive predictions
3. **Failure slices** — Which passenger groups the model gets wrong most often
4. **Gap analysis** — Overfitting vs underfitting diagnosis

The "data vs model" blame is the most important insight for deciding what to fix next.

In [ ]:
debug_result = doctor.debug_model()

print(f"\n{'='*40}")
print("DEBUG RESULTS")
print(f"{'='*40}")
print(f"Train score: {debug_result.train_score:.4f}")
print(f"Test score:  {debug_result.test_score:.4f}")
print(f"Gap:         {debug_result.gap:.4f}")

# Interpret the gap
gap = debug_result.gap
if gap > 0.15:
    print("\n🔴 VERDICT: Overfitting — model memorised training data")
    print("   → Try: more data, regularisation, simpler model")
elif gap < -0.05:
    print("\n🔴 VERDICT: Underfitting — model too simple")
    print("   → Try: more features, complex model, less regularisation")
else:
    print("\n🟢 VERDICT: Healthy generalisation")

### Feature Importances

Which columns does the model rely on most? Unexpected top features can indicate data leakage.

In [ ]:
# Feature importance from debug result
if debug_result.feature_importances is not None:
    fi = debug_result.feature_importances
    print("Top feature importances:")
    print(fi.sort_values(ascending=False).head(10))
else:
    print("Feature importances: use doctor.feature_impact() for counterfactual analysis")

# Counterfactual feature impact (most robust method)
print("\nCounterfactual feature impact (score drop when feature removed):")
impact = doctor.feature_impact(top_n=8)
for feat, drop in sorted(impact.items(), key=lambda x: -x[1]):
    bar = '█' * int(drop * 100)
    print(f"  {feat:20s}  {drop:.4f}  {bar}")

### Step 7 — improve()

Suggestions are **data-aware** and **prioritised**:
- `[HIGH]` — Biggest expected gain. Do these first.
- `[MEDIUM]` — Moderate improvement.
- `[LOW]` — Polish.

Suggestions are generated from all previous steps — health, validation, debug results — not generic advice.

In [ ]:
improvement_report = doctor.improve()

# Also get structured recommendations
print("\n--- Recommended next actions ---")
actions = doctor.recommend_actions()
for i, action in enumerate(actions[:5], 1):
    print(f"  {i}. {action}")

### Step 8 — Trust Score (Production Readiness)

The Trust Score answers: **"Can I safely deploy this model?"**

It's a weighted combination:
```
trust_score = 0.40 × accuracy
            + 0.25 × robustness     (minority class performance)
            + 0.20 × calibration    (predicted confidence vs actual accuracy)
            + 0.15 × certainty      (1 - fraction of uncertain predictions)
```

**Threshold:** Score ≥ 75 = production-ready.

In [ ]:
trust = doctor.trust_score()

print(f"\n{'='*40}")
print("PRODUCTION READINESS")
print(f"{'='*40}")
if trust.score >= 75:
    print(f"✅ Trust Score: {trust.score:.0f} / 100 — PRODUCTION READY")
elif trust.score >= 60:
    print(f"🟡 Trust Score: {trust.score:.0f} / 100 — Needs improvement before production")
else:
    print(f"🔴 Trust Score: {trust.score:.0f} / 100 — NOT production ready")

### Pipeline Confidence Score

A single holistic score (0–100) combining health + validation + training + debug results.

In [ ]:
confidence = doctor.pipeline_confidence()
print(f"\nPipeline Confidence: {confidence} / 100")

### Generate Full Report

In [ ]:
report_path = doctor.report(output_path="intermediate_titanic_report.html")
print(f"📄 Full report: {report_path}")

from IPython.display import IFrame, display
display(IFrame(src='intermediate_titanic_report.html', width='100%', height='600px'))

## Before vs After: auto_improve()

`auto_improve()` is a 3-step automated loop:
1. Baseline train
2. Apply safe fixes
3. Retrain and compare

It shows you the exact score delta from the fixes.

In [ ]:
# Start fresh for a clean before/after comparison
doctor2 = DataDoctor()
doctor2.fit(df, target="Survived")

print("Running auto_improve() — baseline → fix → retrain → compare")
print("=" * 60)
comparison = doctor2.auto_improve(tune=False)

print(f"\n📈 Score delta: {comparison.score_delta:+.4f}")
if comparison.score_delta > 0:
    print(f"✅ Improvement: +{comparison.score_delta:.4f} ({comparison.score_delta*100:.1f}% gain)")
else:
    print("ℹ️  No gain from safe fixes on this dataset (already clean enough)")

## Export Your Model

Save the trained model as a `.joblib` file you can load in production.

In [ ]:
# Export the trained model
model_path = doctor.export_model(path="titanic_model.joblib")
print(f"✅ Model exported: {model_path}")

# Verify it loads correctly
import joblib
loaded_pipeline = joblib.load(model_path)
print(f"✅ Model loaded successfully: {type(loaded_pipeline).__name__}")

# Make a prediction on a sample passenger
sample = df.drop(columns=['Survived']).iloc[[0]]
# Note: loaded model includes all preprocessing steps
print(f"\nSample passenger prediction demo ready")

## 🎯 Exercises

**Exercise 1:** Check dataset difficulty
```python
difficulty = doctor.dataset_difficulty()
print(f"Dataset difficulty: {difficulty:.3f}")  # 0 = easy, 1 = impossible
```

**Exercise 2:** Add a custom validation check
```python
def check_sex_column(df, target):
    if 'Sex' not in df.columns:
        return ["Missing 'Sex' column — critical for Titanic prediction"]
    return []

doctor.add_check(check_sex_column, name="sex_check")
doctor.validate()  # custom check runs as part of validate()
```

**Exercise 3:** Try with tuning enabled
```python
doctor3 = DataDoctor()
doctor3.fit(df, target="Survived")
doctor3.fix(safe=True)
tuned = doctor3.train(tune=True, n_iter=20)
print(f"Tuned: {tuned.test_score:.4f}  Params: {tuned.best_params}")
```

---
## What's Next?

| Notebook | Level | What you'll learn |
|----------|-------|-------------------|
| [Basic (5 min)](demo_basic.ipynb) | 🟢 Basic | fit → health → train → report |
| **You are here** | 🟡 Intermediate | Full pipeline + debug + trust score |
| [Advanced (30 min)](demo_advanced.ipynb) | 🔴 Advanced | Tuning + custom models + feature impact + codegen |
| [Quick Start](quickstart_tabular.ipynb) | ⚡ Reference | All 8 steps in one clean notebook |

---
*KaizenStat v0.5.1 · [GitHub](https://github.com/masuddarrahaman/KaizenStat-Library) · MIT License*